Here I am about to do a model which is gonna make a classification of Credit Card Fraud Detection with some models

In [34]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

df = pd.read_csv("creditcard.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [35]:
# here we need to check our target feature is imbalanced
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True) * 100)

Class
0    284315
1       492
Name: count, dtype: int64
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [36]:
# So dataset is clean enough, now we split it to target and features(X and y)
X = df.drop('Class', axis=1)
y = df['Class']

In [37]:
# Using train test split because we need a part to train our model and check it when it's ready
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
# Using smote cause of class imbalance it helps to keep balanced class
smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42,
    k_neighbors=5
)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before Fraud was: {y_train.sum()}, Normal: {(y_train==0).sum()}")
print(f"After Fraud is: {y_train_resampled.sum()}, Normal: {(y_train_resampled==0).sum()}")

Before Fraud was: 394, Normal: 227451
After Fraud is: 22745, Normal: 227451


I chose XGBoost Classifier because it handles imbalanced datasets 
well, supports early stopping to prevent overfitting, and is 
computationally efficient for large tabular data.

In [39]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='aucpr', # it's good for imbalance
    early_stopping_rounds=20,
    random_state=42
)

model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_test, y_test)],
    verbose=50
)

[0]	validation_0-aucpr:0.57736
[25]	validation_0-aucpr:0.71516


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [45]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud'], zero_division=0))
print(f"ROC AUC:{roc_auc_score(y_test, y_proba):.4f} Area Under the Curve - measures the model's ability to distinguish between fraud and normal transactions")
print(f"Avg Precision:{average_precision_score(y_test, y_proba):.4f} Precision measures how many of the transactions predicted as fraud were actually fraud")

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.54      0.89      0.67        98

    accuracy                           1.00     56962
   macro avg       0.77      0.94      0.84     56962
weighted avg       1.00      1.00      1.00     56962

ROC AUC:0.9865 Area Under the Curve - measures the model's ability to distinguish between fraud and normal transactions
Avg Precision:0.8185 Precision measures how many of the transactions predicted as fraud were actually fraud


In [41]:
lgbm_model = LGBMClassifier(
    n_estimators=500,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.05,
    class_weight='balanced', # for work with smote
    random_state=42,
    verbose=-1
)

lgbm_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)]
)

Training until validation scores don't improve for 20 rounds
[50]	valid_0's binary_logloss: 0.0692201
[100]	valid_0's binary_logloss: 0.018867
[150]	valid_0's binary_logloss: 0.00865184
[200]	valid_0's binary_logloss: 0.00525205
[250]	valid_0's binary_logloss: 0.00404672
[300]	valid_0's binary_logloss: 0.00357118
[350]	valid_0's binary_logloss: 0.0033688
[400]	valid_0's binary_logloss: 0.0033056
Early stopping, best iteration is:
[406]	valid_0's binary_logloss: 0.00329916


LGBMClassifier(class_weight='balanced', learning_rate=0.05, n_estimators=500,
               random_state=42, verbose=-1)

In [42]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',  # for work with smote
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_resampled, y_train_resampled)

RandomForestClassifier(class_weight='balanced', max_depth=10,
                       min_samples_leaf=5, n_jobs=-1, random_state=42)

In [46]:
models = {
    'XGBoost'      : model,
    'LightGBM'     : lgbm_model,
    'Random Forest': rf_model
}

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)
    print(f"{name} — ROC AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud'], zero_division=0))

XGBoost — ROC AUC: 0.9865
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.54      0.89      0.67        98

    accuracy                           1.00     56962
   macro avg       0.77      0.94      0.84     56962
weighted avg       1.00      1.00      1.00     56962

LightGBM — ROC AUC: 0.9835
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.81      0.86      0.83        98

    accuracy                           1.00     56962
   macro avg       0.90      0.93      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Random Forest — ROC AUC: 0.9865
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.54      0.89      0.67        98

    accuracy                           1.00     56962
   macro avg       0.77      0.94      0.84